# NER for geographical place names

This notebook demonstrates use of a vocabulary-driven Named Entity Recognition (NER) pipeline component for use with the [spaCy NLP library](https://spacy.io/) to locate place names within text passages. The `geonames_ruler` component performs NER producing a list of spans identifying the textual positions of matches within the input text. It utilizes NLP functionality to restrict matches to proper nouns and recue ambiguous matches (e.g. "*Wells*" not "*wells*"; "*Street*" not "*street*"). 

This example uses a suite of pipeline components to maximise flexibility and reusability, allowing custom components to be assembled into the [spaCy NLP library](https://spacy.io/) pipeline architecture and combined as required. The components and functionality demonstrated here may also be incorporated into a larger application.

## Data sources
The `geonames_ruler` custom pipeline component identifies place names originating from the following data sources. It does not currently look for named features (e.g. mountains, lakes or rivers), though these could be accommodated if seen as a specific requirement:

* GeoNames [admin1CodesASCII.txt](https://download.geonames.org/export/dump/readme.txt) file - "names in English for admin divisions"
* GeoNames [admin2Codes.txt](https://download.geonames.org/export/dump/readme.txt) file - "names for administrative subdivisions"
* GeoNames [cities500.zip](https://download.geonames.org/export/dump/readme.txt) file - "cities with a population > 500"

The [GeoNames](https://www.geonames.org/) data files listed above are available for download under a [Creative Commons Attribution 4.0 License](https://creativecommons.org/licenses/by/4.0/) from https://download.geonames.org/export/dump/

The use of [GeoNames data](https://download.geonames.org/export/dump/) in the pipeline component facilitates **semantic linking** by providing identifiers for the place names found within the input text. 

## Configuration
Configuration of the `geonames_ruler` component is by a list of ISO country code(s), for example to restrict to place names within the United Kingdom use `["GB"]`- this is actually the default value if no country codes are supplied. The configuration parameter expects a list - so to include multiple countries use e.g. `["GB", "FR"]`. For a full list of country codes see the [GeoNames country codes list](https://download.geonames.org/export/dump/countryInfo.txt). Restricting the component only to specific country code(s) can reduce (though not entirely eliminate) ambiguity. Any ambiguous names (e.g. *Newport*) in the examples below would display with multiple "PLACE" tags in the marked up text passage. The listing of the identified spans then shows the GeoNames identifiers for each of the alternative matches.

In [13]:
%%capture
import warnings
# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)
warnings.filterwarnings(action='ignore', category=FutureWarning)

# install prerequisites for this demonstration
%pip install -U spacy # spaCy NLP library
%sx python -m spacy download en_core_web_sm # English language trained pipeline 

# dependencies used by subsequent code cells
import spacy # NLP processing library
from spacy.tokens import Doc # main class for pipeline component I/O
from spacy import displacy # for visualisation of resultant marked up text
from IPython.display import display, HTML # for displaying results in this notebook
import pandas as pd  # for DataFrame display
from components import GeoNamesRuler # custom spaCy pipeline component for geospatial entity recognition
from decorators import run_once # for caching results of expensive function calls

@run_once
def create_configured_pipeline(config: dict[str, Any]={}) -> Language:    
    # set up default base IE pipeline (English)
    nlp = spacy.load("en_core_web_sm")  
    #nlp = spacy.load("en_core_web_sm", disable=["ner"]) 
    nlp.add_pipe("text_normalizer", first=True)
    nlp.add_pipe("geonames_ruler", last=True, config={"country_codes": ["GB"]})       
    nlp.add_pipe("child_span_remover", last=True) 
    return nlp 


# for inline display of marked up input text     
def display_highlighted(doc: Doc) -> None:    
    displacy.render(
        docs = doc, 
        style = "span", 
        jupyter = True, 
        options = { 
            "spans_key": "rematch",
            "colors": { 
                "GPE": "darkgreen",
                "PLACE": "palegreen"
            }
        } 
    )


# for tabular display of identified spans
def display_spans_table(doc: Doc) -> None:
    # get identified spans from the spaCy Doc object
    spans = doc.spans.get("rematch", [])

    # create DataFrame with required columns
    df = pd.DataFrame([{
        "start": span.start_char,
        "end": span.end_char,
        "token_start": span.start,
        "token_end": span.end - 1,            
        "label": span.label_,
        "id": span.id_,
        "text": span.text
        } for span in spans])

    # render DataFrame as html table
    display(HTML(df.to_html(index=False, border=True)))


## Usage example 1
The following Python code demonstrates some basic usage of the *geonames_ruler* custom pipeline component. It is demonstrated here in conjunction with the default spaCy NER functionality to be able to visually compare the results of using the two methods e.g. in the result and output of the example below, many of the mentioned place names (e.g. "England") do not seem to be identified by the default spaCy functionality. 

Note *Leicestershire* is identified via two routes - using the default spaCy NER functionality (tagged as "GPE" - Geo-Politial Entity - without an associated identifier), and via the custom pipeline component (tagged as "PLACE" - with an associated GeoNames identifier). In the `create_configured_pipeline` function above an optional filter component `child_span_remover` is used to suppress overlapping spans (actually a wrapper for [spacy.util.filter_spans](https://spacy.io/api/top-level#util.filter_spans)). Without filtering, *Rhondda* would appear as a match within *Rhondda Cynon Taf* - depending on the ultimate use-case you may choose to filter the spans or not.

In [14]:
nlp = create_configured_pipeline()

# run full IE pipeline on example test text
input_text = "Ashby-de-la-Zouch is a town located in Leicestershire (England). The USW university campus is located within Rhondda Cynon Taf about 10 miles North-West of Cardiff (near Pontypridd)."
doc = nlp(input_text)

# (optionally) also include any place entities identified by
# the default spaCy NER functionality for subsequent display 
for ent in filter(lambda e: e.label_ == "GPE", doc.ents):
    doc.spans["rematch"].append(ent)

# display the NER results
display_highlighted(doc)
display_spans_table(doc)


start,end,token_start,token_end,label,id,text
0,23,0,6,PLACE,http://sws.geonames.org/2656970/,Ashby - de - la - Zouch
45,59,12,12,PLACE,http://sws.geonames.org/2644667/,Leicestershire
61,68,14,14,PLACE,http://sws.geonames.org/6269131/,England
116,133,24,26,PLACE,http://sws.geonames.org/3333247/,Rhondda Cynon Taf
165,172,34,34,PLACE,http://sws.geonames.org/3333241/,Cardiff
179,189,37,37,PLACE,http://sws.geonames.org/2640104/,Pontypridd
45,59,12,12,GPE,,Leicestershire


## Usage example 2
The test data used here is a set of 20 example records provided by the Archaeology Data Service (ADS) for use in ATRIUM task T4.1 (text based workflows). The original data file is a google sheet located in the (privately shared) ATRIUM shared Google drive - under ATRIUM / WP4 / T4.1 / Subtask 4.1.2 / ADS-UoY_samples / oasis_reports_july_2024 / report_metadata. The results immediately following this source code illustrates HTML rendered output of the pipeline, but the results may be expressed in many different formats. Again the custom pipeline components are used in conjunction with the default spaCy NER functionality to be able to visually compare the results of using the two approaches.

In [20]:
# read a set of test records from source (CSV) data file 
# source data file columns are "file", "title", "abstract", "doi"
#file_path = "./data/ads/report_metadata.csv" 
file_path = "./data/oasis/journals_july_2024/journal_metadata.csv"

# read CSV input file
df = pd.read_csv(file_path, skip_blank_lines=True)
test_records = df.to_dict(orient="records")   

# set up default base IE pipeline (English)
nlp = spacy.load("en_core_web_sm")   

# add custom pipeline IE component(s) to the end of the default pipeline
nlp.add_pipe("geonames_ruler", last=True, config={"country_codes": ["GB"]})  

# (optionally) suppress matches on encompassed spans 
nlp.add_pipe("child_span_remover", last=True) 
   
# process each test record using the custom pipeline  
for record in test_records:
    # extract record fields for processing  
    #test_id = record.get("doi","").strip()
    #test_title = record.get("title","").strip()
    #test_text = record.get("abstract","").strip()

    test_id = str(record.get("object_id","")).strip()
    test_title = record.get("title","").strip()
    test_text = record.get("abstract","").strip()
    
    # run IE pipeline on title and text combined
    input_text = f"{test_title}. {test_text}"
    doc = nlp(input_text)

    # (optionally) add any place entities identified by the
    # default spaCy NER functionality for subsequent display. 
    # Note you may wish to omit this step; these will not have 
    # identifiers as they did not originate from GeoNames.
    for ent in filter(lambda e: e.label_ == "GPE", doc.ents):
        doc.spans["rematch"].append(ent) 

    # the identified spans are stored here
    # spans = doc.spans.get("rematch", [])

    # display record identifier 
    display(HTML(f"<strong>{test_id}</strong><br>"))
    
    # display input text highlighted with identified places
    display_highlighted(doc)

    # display table with identifiers and locations within the text
    display_spans_table(doc)
    
    # display horizontal rule before next record
    display(HTML("<hr>"))           

start,end,token_start,token_end,label,id,text
665,679,121,121,PLACE,http://sws.geonames.org/2641235/,Northumberland


start,end,token_start,token_end,label,id,text
26,30,5,5,PLACE,http://sws.geonames.org/2634877/,Wall
97,101,21,21,PLACE,http://sws.geonames.org/2634877/,Wall
191,195,37,37,PLACE,http://sws.geonames.org/2634877/,Wall


start,end,token_start,token_end,label,id,text
15,34,3,5,PLACE,http://sws.geonames.org/3333174/,Newcastle upon Tyne
219,238,33,35,PLACE,http://sws.geonames.org/3333174/,Newcastle upon Tyne
833,839,135,135,PLACE,http://sws.geonames.org/2636671/,Street
15,24,3,3,GPE,,Newcastle
219,228,33,33,GPE,,Newcastle


start,end,token_start,token_end,label,id,text
120,139,25,27,PLACE,http://sws.geonames.org/3333174/,Newcastle upon Tyne
216,225,44,44,PLACE,http://sws.geonames.org/6695976/,Newcastle
120,129,25,25,GPE,,Newcastle
135,139,27,27,GPE,,Tyne
216,225,44,44,GPE,,Newcastle


start,end,token_start,token_end,label,id,text
44,63,9,11,PLACE,http://sws.geonames.org/3333174/,Newcastle upon Tyne
44,53,9,9,GPE,,Newcastle
59,63,11,11,GPE,,Tyne


...

start,end,token_start,token_end,label,id,text
18,24,4,4,PLACE,http://sws.geonames.org/2647007/,Hexham
18,24,4,4,GPE,,Hexham


start,end,token_start,token_end,label,id,text
15,24,2,2,PLACE,http://sws.geonames.org/6695976/,Newcastle


start,end,token_start,token_end,label,id,text
18,24,3,3,PLACE,http://sws.geonames.org/2647007/,Hexham


start,end,token_start,token_end,label,id,text
57,71,9,9,PLACE,http://sws.geonames.org/2641235/,Northumberland
272,279,44,44,PLACE,http://sws.geonames.org/6269131/,England
284,292,46,46,PLACE,http://sws.geonames.org/2638360/,Scotland
272,279,44,44,GPE,,England
284,292,46,46,GPE,,Scotland


AttributeError: 'float' object has no attribute 'strip'